In [2]:
import sys
import re
import time
from collections import defaultdict
import string
import requests
import pandas as pd
from requests import Session
from keybert import KeyBERT

c:\Users\tungo\Projects\qcaass-matching\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
S2_API = "https://api.semanticscholar.org/graph/v1/paper/DOI:{doi}"
S2_FIELDS = "title,abstract,venue,publicationVenue,externalIds"

PREFIX_MAP = {
    "10.1145": "ACM",
    "10.1109": "IEEE",
    "10.1007": "Springer-Verlag",
}

TARGET_PUBLISHERS = set(PREFIX_MAP.values())

REQUEST_DELAY = 1.0


In [3]:
def fetch_s2_record(doi: str, session: Session):
    # query Semantic Scholar for single DOI
    url = S2_API.format(doi=doi)
    try:
        resp = session.get(url, params={"fields": S2_FIELDS}, timeout=15)
    except requests.RequestException as err:
        print(f"Request error for {doi}:{err}")
        return None
    
    if resp.status_code == 404:
        print(f"{doi} not found in Semantic Scholar.")
        return None
    if resp.status_code == 429:
        print("Rate limited reached, sleep 10s and retrying once")
        time.sleep(10)
        try:
            resp = session.get(url, params={"fields": S2_FIELDS}, timeout=15)
        except requests.RequestException as err:
            print(f"Retry failed for {doi}: {e}")
            return None
    if resp.status_code != 200:
        print(f"HTTP {resp.status_code} for {doi}")
        return None

    try:
        return resp.json()
    except ValueError:
        print(f"Bad Json for {doi}")
        return None

def normalize_abstract(raw_abstract: str):
    """Strip markup, collapse whitespace, lowercase, strip punctuation."""
    if not raw_abstract:
        return ""
 
    text = raw_abstract
 
    text = re.sub(r"<[^>]+>", " ", text)          # HTML/XML tags
    text = re.sub(r"\$.*?\$", " ", text)          # inline LaTeX math
    text = re.sub(r"\\[a-zA-Z]+\{[^}]*\}", " ", text)  # \command{...}
    text = re.sub(r"\\[a-zA-Z]+", " ", text)      # bare \command
 
    # Collapse whitespace / line breaks
    text = re.sub(r"\s+", " ", text).strip()
 
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
 
    # Collapse any double spaces left behind by punctuation removal
    text = re.sub(r"\s+", " ", text).strip()
    return text

def resolve_publisher(doi:str):
    prefix = doi.split("/")[0]
    return PREFIX_MAP.get(prefix)

In [ ]:
INPUT = 'data/Final Set- WithQAScores- WithoutStudyID(Remove Duplicated).csv'
OUTPUT = 'out/un_matched_with_abstract.csv'

session = Session()
session.headers.update({"User-Agent": "explore-abstract"})

output_rows = []
skipped = []
not_found = []
empty_abstract = []

un_matched = pd.read_csv(INPUT, dtype=str, engine='python', on_bad_lines='warn')

for _, row in un_matched.iterrows():
    doi = row['DOI']
    # if resolve_publisher(doi) is None:
    #     print(f"DOI {doi} is skipped.")
    #     skipped.append(doi)
    #     continue

    s2_record = fetch_s2_record(doi, session)
    if s2_record is None:
        # print(f"DOI {doi}'s record is not found.")
        not_found.append(doi)
        time.sleep(REQUEST_DELAY)
        continue
    raw_abstract = s2_record.get("abstract")
    if raw_abstract is not None:
        normalized_abstract = normalize_abstract(raw_abstract)

        output_rows.append({
            "StudyID": row['StudyID'],
            "Title": row['Title'],
            "DOI": row['DOI'],
            # "Publisher": resolve_publisher(doi),
            "Abstract_Normalized": normalized_abstract,
        })
        print(f"Add abstract for {doi}")
    else:
        empty_abstract.append(doi)
        output_rows.append({
            "StudyID": row['StudyID'],
            "Title": row['Title'],
            "DOI": row['DOI'],
            # "Publisher": resolve_publisher(doi),
            "Abstract_Normalized": "",
        })
    time.sleep(REQUEST_DELAY)

populated_df = pd.DataFrame(
    output_rows,
    columns=["StudyID", "Title", "DOI", "Abstract_Normalized"],
)

print("\n--- Summary ---")
print(f"Total input rows: {len(un_matched)}")
print(f"Kept (ACM/Springer/IEEE by DOI prefix): {len(output_rows)}")
print(f"Skipped (DOI prefix not in target list): {len(skipped)}")
print(f"Empty abstract: {len(empty_abstract)}")
print(f"Not found in Semantic Scholar: {len(not_found)}")

Add abstract for 10.1007/978-3-030-01461-2_8
10.1201/9788770046336-2 not found in Semantic Scholar.
Add abstract for 10.1007/s11227-025-07969-2
Add abstract for 10.1145/3659996.3660035
Add abstract for 10.1145/3688856
Add abstract for 10.1109/qce57702.2023.10213
Add abstract for 10.1145/3731599.3767548
Add abstract for 10.1109/SCW63240.2024.00205
Add abstract for 10.1038/s41566-024-01403-4
Add abstract for 10.1088/2058-9565/ab9acb
10.1145/3672608.37079 not found in Semantic Scholar.
Add abstract for 10.1109/CCGRID64434.2025.00069
Add abstract for 10.1109/DAC56929.2023.10247886
Add abstract for 10.1145/3663531.3664752
Add abstract for 10.1145/3643667.3648224
10.1007/978-3-031-85884-0_12 not found in Semantic Scholar.
Add abstract for 10.1145/3462670
Add abstract for 10.1145/3613424.3614300
Add abstract for 10.1016/j.future.2024.06.058
Add abstract for 10.1109/QSW59989.2023.00013
Add abstract for 10.1002/spe.3331
Add abstract for 10.1109/tqe.2023.3275868
10.1007/978-3-031-62076-8_10 not 

In [5]:
populated_df.to_csv('out/un_matched_abstract_populated.csv', index=False)

In [4]:
populated_df = pd.read_csv("un_matched_abstract_populated(manual_inspected).csv", dtype=str, encoding="latin-1").fillna("")
docs = populated_df['Abstract_Normalized'].tolist()
print(f"Abstracts: {len(docs)}")

kw_model = KeyBERT('all-MiniLM-L12-v2')

scores = defaultdict(float)
for doc in docs:
    for phrase, score in kw_model.extract_keywords(
        doc,
        keyphrase_ngram_range=(1, 4),
        stop_words='english',
        use_mmr=True, diversity=0.65,
        top_n=10,
    ):
        scores[phrase] += score          # aggregate across the corpus

top10 = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:10]
print("\nTop 10 phrases across all abstracts:")
for phrase, score in top10:
    print(f"{phrase:80s} {score:.3f}")


Abstracts: 87


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4304.67it/s]



Top 10 phrases across all abstracts:
stateoftheart                                                                    0.938
quantum software service                                                         0.867
service oriented quantum computing                                               0.865
quantum services automatic generation                                            0.838
debugging faulty quantum programs                                                0.824
quantum software developers qsandbox                                             0.824
developing hybrid quantumclassical software                                      0.821
algorithms appropriate quantum computers                                         0.816
existing quantum programming                                                     0.813
highlevel quantum programming language                                           0.809
